# 06 — Sampling and Temperature

**Description:** Turn next-token probabilities into generated text, compare greedy decoding with sampling, and control randomness using temperature.
**Level:** Beginner
**Tags:** Language Models, Generation, Sampling, Temperature, Decoding

Notebook 05 ended with a probability distribution over possible next tokens. A distribution is not yet text: generation needs a rule for selecting one token and then repeating the process.

By the end, you will be able to:

- distinguish greedy decoding from categorical sampling;
- verify sampling behavior with repeated experiments;
- apply temperature correctly to logits;
- explain how temperature changes uncertainty without changing token ranking;
- generate complete sequences one token at a time; and
- use top-k and top-p filtering as optional sampling controls.

In [ ]:
from collections import Counter

import matplotlib.pyplot as plt
import numpy as np

np.set_printoptions(precision=4, suppress=True)
plt.style.use("seaborn-v0_8-whitegrid")
rng = np.random.default_rng(7)

## 1. Begin with next-token logits

Suppose the model has processed the context `the robot` and produced one logit for every vocabulary token. As in Notebook 05, stable softmax converts these scores into probabilities.

In [ ]:
vocabulary = np.array(["learns", "writes", "runs", "dreams", ".", "<EOS>"])
logits = np.array([2.4, 1.8, 0.9, 1.2, -0.5, -1.0])

def softmax(values, axis=-1):
    values = np.asarray(values, dtype=float)
    shifted = values - np.max(values, axis=axis, keepdims=True)
    exponentials = np.exp(shifted)
    return exponentials / exponentials.sum(axis=axis, keepdims=True)

probabilities = softmax(logits)
for token, logit, probability in zip(vocabulary, logits, probabilities):
    print(f"{token:8s} | logit={logit:5.2f} | probability={probability:6.2%}")

## 2. Greedy decoding chooses the maximum

The simplest decoding rule is **greedy decoding**: select the token with the highest probability. Because softmax preserves order, taking the largest logit gives the same result and avoids computing probabilities.

Greedy decoding is deterministic. For the same model state, it always makes the same choice.

In [ ]:
best_logit_id = np.argmax(logits)
best_probability_id = np.argmax(probabilities)

print("argmax logit:      ", vocabulary[best_logit_id])
print("argmax probability:", vocabulary[best_probability_id])
print("same ID:           ", best_logit_id == best_probability_id)

## 3. Sampling treats probabilities as chances

**Categorical sampling** randomly selects one token according to the distribution. A token with probability 40% should be chosen about 40% of the time over many independent samples.

Sampling is not uniform randomness. High-probability tokens remain more likely; lower-probability tokens simply retain a chance to appear.

In [ ]:
def sample_token(tokens, probabilities, generator=rng):
    token_id = generator.choice(len(tokens), p=probabilities)
    return str(tokens[token_id])

samples = [sample_token(vocabulary, probabilities) for _ in range(12)]
print("greedy: ", vocabulary[np.argmax(probabilities)])
print("samples:", samples)

### Verify sampling empirically

A small sample can look uneven by chance. With thousands of draws, observed frequencies should approach the target probabilities. They will not be exactly equal because sampling contains random variation.

In [ ]:
number_of_draws = 20_000
drawn_ids = rng.choice(len(vocabulary), size=number_of_draws, p=probabilities)
observed = np.bincount(drawn_ids, minlength=len(vocabulary)) / number_of_draws

print(f"{'token':8s} {'expected':>10s} {'observed':>10s}")
for token, expected, actual in zip(vocabulary, probabilities, observed):
    print(f"{token:8s} {expected:10.2%} {actual:10.2%}")

### Visualize expected and observed frequencies

The two bars for each token should nearly overlap. Increase or decrease `number_of_draws` and observe how sampling noise changes.

In [ ]:
positions = np.arange(len(vocabulary))
width = 0.38
fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(positions - width / 2, probabilities, width, label="expected", color="#4C78A8")
ax.bar(positions + width / 2, observed, width, label="observed", color="#F58518")
ax.set_xticks(positions, vocabulary, rotation=30)
ax.set(title=f"Sampling frequencies over {number_of_draws:,} draws", ylabel="Frequency", ylim=(0, 0.6))
ax.legend()
plt.show()

## 4. Seeds make experiments reproducible

A pseudorandom-number generator produces a deterministic sequence from an initial **seed**. Reusing the same seed reproduces the same samples, which is valuable for debugging and teaching. Changing the seed changes the random sequence, not the underlying probabilities.

In [ ]:
def sample_with_seed(seed, count=8):
    local_rng = np.random.default_rng(seed)
    return local_rng.choice(vocabulary, size=count, p=probabilities).tolist()

print("seed 10, first run: ", sample_with_seed(10))
print("seed 10, second run:", sample_with_seed(10))
print("seed 11:            ", sample_with_seed(11))

## 5. Temperature reshapes the distribution

Temperature $T$ is applied **before softmax** by dividing the logits:

$$P_T(i) = \operatorname{softmax}(z_i / T), \qquad T > 0$$

- $T < 1$ increases logit gaps, creating a sharper distribution.
- $T = 1$ leaves the original distribution unchanged.
- $T > 1$ decreases gaps, creating a flatter distribution.

Temperature does not change the ranking of finite logits. It changes how strongly the model prefers the leaders.

In [ ]:
def probabilities_with_temperature(logits, temperature=1.0):
    if temperature <= 0:
        raise ValueError("temperature must be greater than zero")
    return softmax(np.asarray(logits) / temperature)

for temperature in [0.25, 0.5, 1.0, 2.0, 5.0]:
    probs = probabilities_with_temperature(logits, temperature)
    print(f"T={temperature:>4}: {np.round(probs, 3)}")

### Compare temperature visually

At low temperature, the highest logit dominates. At high temperature, probability mass spreads toward other tokens. In the limit $T \to 0^+$, the distribution approaches greedy choice. In the limit $T \to \infty$, it approaches a uniform distribution over finite logits.

In [ ]:
temperatures = [0.25, 0.5, 1.0, 2.0, 5.0]
fig, axes = plt.subplots(len(temperatures), 1, figsize=(8, 10), sharex=True, sharey=True)
for ax, temperature in zip(axes, temperatures):
    probs = probabilities_with_temperature(logits, temperature)
    ax.bar(vocabulary, probs, color="#4C78A8")
    ax.set_ylabel(f"T={temperature}")
    ax.set_ylim(0, 1)
axes[-1].tick_params(axis="x", rotation=30)
axes[0].set_title("The same logits at different temperatures")
plt.tight_layout()
plt.show()

## 6. Why temperature belongs before softmax

Dividing probabilities by temperature and renormalizing does nothing: the shared scaling cancels. Temperature must scale logits before exponentiation, where it changes their relative odds.

In [ ]:
temperature = 2.0
correct = softmax(logits / temperature)
incorrect = (probabilities / temperature) / (probabilities / temperature).sum()

print("original:                 ", probabilities)
print("correct: softmax(logits/T):", correct)
print("incorrect: scale probs:    ", incorrect)
print("incorrect equals original:", np.allclose(incorrect, probabilities))

## 7. Temperature changes odds predictably

For tokens $a$ and $b$, temperature changes the odds ratio to:

$$\frac{P_T(a)}{P_T(b)} = e^{(z_a-z_b)/T}$$

Low temperature magnifies the logit gap; high temperature shrinks it. This provides a precise explanation for sharpening and flattening.

In [ ]:
a, b = 0, 1  # learns versus writes
gap = logits[a] - logits[b]
for temperature in [0.25, 0.5, 1.0, 2.0, 5.0]:
    probs = probabilities_with_temperature(logits, temperature)
    measured_odds = probs[a] / probs[b]
    expected_odds = np.exp(gap / temperature)
    print(f"T={temperature:>4} | measured={measured_odds:8.3f} | expected={expected_odds:8.3f}")

## 8. Entropy measures uncertainty

Entropy summarizes how spread out a distribution is:

$$H(P) = -\sum_i P(i) \log P(i)$$

Low entropy means probability mass is concentrated on a few choices. High entropy means the distribution is more even. Temperature usually increases entropy as it increases, provided the logits are finite.

In [ ]:
def entropy(probabilities, eps=1e-12):
    probabilities = np.asarray(probabilities)
    return float(-np.sum(probabilities * np.log(np.maximum(probabilities, eps))))

temperature_grid = np.geomspace(0.1, 10.0, 100)
entropies = [entropy(probabilities_with_temperature(logits, t)) for t in temperature_grid]

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(temperature_grid, entropies, color="#E45756")
ax.axhline(np.log(len(vocabulary)), color="gray", linestyle="--", label="uniform maximum")
ax.set(xscale="log", title="Temperature and distribution entropy",
       xlabel="Temperature", ylabel="Entropy (nats)")
ax.legend()
plt.show()

## 9. Observe temperature through samples

Repeated draws make the tradeoff concrete. Low temperatures repeatedly select the leading token. Higher temperatures produce more variety, including tokens the model originally considered unlikely.

In [ ]:
for temperature in [0.25, 0.7, 1.0, 2.0, 5.0]:
    probs = probabilities_with_temperature(logits, temperature)
    samples = rng.choice(vocabulary, size=16, p=probs)
    counts = Counter(samples)
    print(f"T={temperature:>4}: {dict(counts)}")

### Your turn: choose a temperature

Change `my_temperature`, inspect the distribution, and sample 20 tokens. There is no universally best temperature: the useful setting depends on the model, prompt, and desired balance between consistency and diversity.

In [ ]:
my_temperature = 0.8  # Edit me
my_probabilities = probabilities_with_temperature(logits, my_temperature)

for token, probability in zip(vocabulary, my_probabilities):
    print(f"{token:8s} {probability:6.2%}")
print("samples:", rng.choice(vocabulary, size=20, p=my_probabilities).tolist())

## 10. Generation repeats prediction and selection

To generate more than one token, a model loops:

1. use the current context to compute next-token logits;
2. apply temperature and softmax;
3. select a token;
4. append it to the context; and
5. stop at `<EOS>` or a length limit.

Our tiny model stores logits for the most recent token. A transformer would compute them from the entire available context.

In [ ]:
generation_vocabulary = np.array([
    "the", "robot", "model", "learns", "writes", "runs",
    "from", "data", "stories", "quickly", ".", "<EOS>"
])

next_token_logits = {
    "<START>": {"the": 3.0},
    "the": {"robot": 2.2, "model": 1.8},
    "robot": {"learns": 2.2, "writes": 1.8, "runs": 1.0},
    "model": {"learns": 2.4, "writes": 1.4},
    "learns": {"from": 2.8, "quickly": 1.3, ".": 0.5},
    "writes": {"stories": 2.5, "quickly": 0.9, ".": 0.6},
    "runs": {"quickly": 2.4, ".": 1.0},
    "from": {"data": 3.0},
    "data": {".": 3.0},
    "stories": {".": 3.0},
    "quickly": {".": 3.0},
    ".": {"<EOS>": 3.0},
}

print("vocabulary size:", len(generation_vocabulary))
print("states with learned transitions:", len(next_token_logits))

### Convert sparse scores into a full logit vector

Tokens absent from a state's dictionary receive a very low score. This keeps the example compact while still producing one logit for every vocabulary entry.

In [ ]:
def logits_for(previous_token):
    scores = next_token_logits.get(previous_token, {"<EOS>": 3.0})
    return np.array([scores.get(token, -10.0) for token in generation_vocabulary])

robot_logits = logits_for("robot")
robot_probabilities = probabilities_with_temperature(robot_logits, 1.0)
for token, probability in zip(generation_vocabulary, robot_probabilities):
    if probability > 0.001:
        print(f"{token:8s} {probability:6.2%}")

### Implement autoregressive generation

Temperature zero is often used as shorthand for greedy decoding, even though the formula `logits / 0` is undefined. Our function handles `temperature=0` as a separate deterministic case.

In [ ]:
def generate(prompt="", temperature=1.0, max_new_tokens=12, generator=rng, show_steps=False):
    generated = prompt.split() if prompt else []
    previous = generated[-1] if generated else "<START>"

    for step in range(max_new_tokens):
        step_logits = logits_for(previous)
        if temperature == 0:
            token_id = int(np.argmax(step_logits))
            probability = 1.0
        elif temperature > 0:
            step_probabilities = probabilities_with_temperature(step_logits, temperature)
            token_id = int(generator.choice(len(generation_vocabulary), p=step_probabilities))
            probability = step_probabilities[token_id]
        else:
            raise ValueError("temperature cannot be negative")

        token = str(generation_vocabulary[token_id])
        if show_steps:
            print(f"step {step + 1:2d} | after {previous!r:9s} → {token!r:9s} ({probability:.1%})")
        if token == "<EOS>":
            break
        generated.append(token)
        previous = token

    return " ".join(generated)

generate("the robot", temperature=1.0, show_steps=True)

## 11. Compare generated text across temperatures

The tiny model's grammar is constrained, so even high-temperature results remain readable. Still, low temperatures repeatedly follow the highest-scoring path, while higher temperatures visit lower-scoring branches more often.

Because each sampled token becomes future context, one random choice can change every later prediction.

In [ ]:
for temperature in [0, 0.3, 0.7, 1.0, 2.0]:
    print(f"\nTemperature {temperature}:")
    for _ in range(5):
        print("  ", generate("the robot", temperature=temperature))

## 12. Optional: top-k filtering

Temperature redistributes probability across all tokens. **Top-k sampling** first keeps only the `k` highest logits, sets all others to negative infinity, and then applies softmax. This prevents the long tail from being sampled.

With `k=1`, top-k becomes greedy decoding. A fixed `k` ignores whether the retained probabilities are concentrated or diffuse.

In [ ]:
def top_k_probabilities(logits, k, temperature=1.0):
    if not 1 <= k <= len(logits):
        raise ValueError("k must be between 1 and the vocabulary size")
    scaled = np.asarray(logits, dtype=float) / temperature
    keep = np.argpartition(scaled, -k)[-k:]
    filtered = np.full_like(scaled, -np.inf)
    filtered[keep] = scaled[keep]
    return softmax(filtered)

for k in [1, 2, 3, len(vocabulary)]:
    probs = top_k_probabilities(logits, k)
    kept = [(token, round(float(p), 3)) for token, p in zip(vocabulary, probs) if p > 0]
    print(f"k={k}: {kept}")

## 13. Optional: top-p or nucleus filtering

**Top-p sampling** keeps the smallest set of highest-probability tokens whose cumulative probability reaches a threshold `p`. The number of retained tokens adapts to the distribution: fewer for a confident prediction and more for an uncertain one.

After filtering, retained probabilities must be renormalized to sum to 1.

In [ ]:
def top_p_probabilities(logits, p=0.9, temperature=1.0):
    if not 0 < p <= 1:
        raise ValueError("p must be in (0, 1]")
    probabilities = probabilities_with_temperature(logits, temperature)
    order = np.argsort(probabilities)[::-1]
    cumulative = np.cumsum(probabilities[order])
    count = np.searchsorted(cumulative, p, side="left") + 1
    keep = order[:count]
    filtered = np.zeros_like(probabilities)
    filtered[keep] = probabilities[keep]
    return filtered / filtered.sum()

for p in [0.5, 0.8, 0.95, 1.0]:
    probs = top_p_probabilities(logits, p)
    kept = [(token, round(float(prob), 3)) for token, prob in zip(vocabulary, probs) if prob > 0]
    print(f"p={p}: {kept}")

## 14. Decoding controls do not improve the model

Temperature, top-k, and top-p change how existing model scores become choices. They do not add knowledge, repair incorrect logits, or make an unsafe continuation safe.

Practical tradeoffs include:

- very low temperature can be repetitive and brittle;
- very high temperature can select incoherent low-score tokens;
- aggressive filtering can remove a good token;
- one random early choice can redirect the continuation; and
- the same setting can behave differently across models and prompts.

Evaluate decoding settings on the actual task rather than treating one value as universally optimal.

## 15. Challenges

1. **Empirical sampling:** Repeat the frequency experiment with 100, 1,000, and 100,000 draws. Measure the largest absolute error each time.
2. **Temperature limits:** Test temperatures `0.01` and `100`. Compare the output with greedy and uniform distributions.
3. **Rank preservation:** Verify programmatically that every positive temperature produces the same token ranking as the original logits.
4. **Entropy target:** Search for a temperature that gives entropy close to 1.0 nat.
5. **Generation paths:** Generate 1,000 sequences at two temperatures and count the distinct results.
6. **Top-k generation:** Extend `generate` with a `top_k` argument.
7. **Sequence probability:** Modify `generate` to return the probability of each chosen token and the product or sum of log probabilities for the complete sequence.
8. **Reasoning:** Explain why comparing raw sequence-probability products favors shorter sequences.

In [ ]:
# Challenge workspace: find a temperature with entropy near a target.
target_entropy = 1.0
candidates = np.geomspace(0.05, 10.0, 1_000)
candidate_entropies = np.array([
    entropy(probabilities_with_temperature(logits, temperature))
    for temperature in candidates
])
best_index = np.argmin(np.abs(candidate_entropies - target_entropy))

print("temperature:", candidates[best_index])
print("entropy:    ", candidate_entropies[best_index])

## Series recap

Across these six notebooks, we followed the core language-model pipeline:

1. text becomes tokens and token IDs;
2. embedding lookup turns IDs into vectors;
3. learned layers transform those vectors into contextual hidden states;
4. unembedding converts a hidden state into vocabulary logits;
5. softmax converts logits into probabilities; and
6. decoding selects one token, appends it, and repeats.

The architecture and training data behind a modern GPT are vast, but generation still reduces to this understandable token-by-token loop.

## Takeaways

- Greedy decoding always chooses the highest logit and is deterministic.
- Categorical sampling selects tokens according to their probabilities.
- Repeated sample frequencies approach the model distribution.
- Temperature divides logits before softmax: low values sharpen and high values flatten.
- Temperature changes uncertainty but preserves token ranking.
- Autoregressive generation feeds every selected token back into the next prediction.
- Top-k and top-p can restrict the candidate set before sampling.
- Decoding settings reshape model behavior but do not improve the underlying model.